# Class 4. Pandas practice

## Plan

<h2>

1) Pandas practice <br>

</h2>

# Imports

In [1]:
import pandas as pd
from pandas.testing import assert_frame_equal

from IPython.core.display import HTML #fancy table formatting
table_css = 'table {align:left;display:block} '
HTML('<style>{}</style>'.format(table_css))

## Pandas practce

### Exercise 1

Table: `scores`

| Column Name | Type    |
|-------------|---------|
| id          | int     |
| score       | decimal |

id is the primary key (column with unique values) for this table.\
Each row of this table contains the score of a game. Score is a floating point value with two decimal places.
 

Write a solution to find the rank of the scores. The ranking should be calculated according to the following rules:

The scores should be ranked from the highest to the lowest.\
If there is a tie between two scores, both should have the same ranking.\
After a tie, the next ranking number should be the next consecutive integer value. In other words, there should be no holes between ranks.\
Return the result table ordered by score in descending order.

The result format is in the following example.

 

Example 1:

Input: \
Scores table:

| id | score |
|----|-------|
| 1  | 3.50  |
| 2  | 3.65  |
| 3  | 4.00  |
| 4  | 3.85  |
| 5  | 4.00  |
| 6  | 3.65  |

Output: 

| score | rank |
|-------|------|
| 4.00  | 1    |
| 4.00  | 1    |
| 3.85  | 2    |
| 3.65  | 3    |
| 3.65  | 3    |
| 3.50  | 4    |


In [2]:
def sol_1(scores: pd.DataFrame) -> pd.DataFrame:
    scores_unique = scores['score'].unique()
    mapping = {}
    for score, rank in zip(sorted(scores_unique, reverse=True), range(1, len(scores)+1)):
        mapping[score] = rank
    df = scores[['score']]
    df['rank'] = list(map(lambda x: mapping[x], df['score']))
    return df.sort_values('score', ascending=False)

In [3]:
def _norm(df: pd.DataFrame) -> pd.DataFrame:
    return df.reset_index(drop=True)


def _case(scores_rows, exp_rows):
    scores = pd.DataFrame(scores_rows, columns=["id", "score"])
    got = sol_1(scores)
    exp = pd.DataFrame(exp_rows, columns=["score", "rank"])
    assert_frame_equal(_norm(got), _norm(exp), check_dtype=False)


def run():
    cases = [
        ("example", lambda: _case(
            [(1, 3.50), (2, 3.65), (3, 4.00), (4, 3.85), (5, 4.00), (6, 3.65)],
            [(4.00, 1), (4.00, 1), (3.85, 2), (3.65, 3), (3.65, 3), (3.50, 4)],
        )),
        ("all_equal", lambda: _case(
            [(1, 1.23), (2, 1.23), (3, 1.23)],
            [(1.23, 1), (1.23, 1), (1.23, 1)],
        )),
        ("strict_desc", lambda: _case(
            [(1, 9.0), (2, 8.0), (3, 7.0)],
            [(9.0, 1), (8.0, 2), (7.0, 3)],
        )),
        ("ties_middle", lambda: _case(
            [(1, 9.0), (2, 8.0), (3, 8.0), (4, 7.0), (5, 6.0)],
            [(9.0, 1), (8.0, 2), (8.0, 2), (7.0, 3), (6.0, 4)],
        )),
        ("ties_top", lambda: _case(
            [(1, 5.0), (2, 5.0), (3, 4.0)],
            [(5.0, 1), (5.0, 1), (4.0, 2)],
        )),
        ("ties_bottom", lambda: _case(
            [(1, 3.0), (2, 2.0), (3, 1.0), (4, 1.0)],
            [(3.0, 1), (2.0, 2), (1.0, 3), (1.0, 3)],
        )),
        ("unsorted_input", lambda: _case(
            [(1, 2.5), (2, 9.5), (3, 9.5), (4, 3.0)],
            [(9.5, 1), (9.5, 1), (3.0, 2), (2.5, 3)],
        )),
        ("many_duplicates", lambda: _case(
            [(1, 4.2), (2, 4.2), (3, 4.2), (4, 1.0), (5, 1.0), (6, 0.5)],
            [(4.2, 1), (4.2, 1), (4.2, 1), (1.0, 2), (1.0, 2), (0.5, 3)],
        )),
        ("negative_scores", lambda: _case(
            [(1, -1.0), (2, -2.0), (3, -2.0), (4, -3.0)],
            [(-1.0, 1), (-2.0, 2), (-2.0, 2), (-3.0, 3)],
        )),
        ("two_rows", lambda: _case(
            [(1, 10.0), (2, 9.99)],
            [(10.0, 1), (9.99, 2)],
        )),
    ]

    failed = 0
    for _, f in cases:
        try:
            f()
        except Exception:
            failed += 1
    print(f'Failed: {failed}')

run()

Failed: 0


### Exercise 2

Table: `activity`


| Column Name  | Type    |
|--------------|---------|
| player_id    | int     |
| device_id    | int     |
| event_date   | date    |
| games_played | int     |

(player_id, event_date) is the primary key (combination of columns with unique values) of this table.\
This table shows the activity of players of some games.\
Each row is a record of a player who logged in and played a number of games (possibly 0) before logging out on someday using some device.

Write a solution to report the fraction of players that logged in again on the day after the day they first logged in, rounded to 2 decimal places. In other words, you need to determine the number of players who logged in on the day immediately following their initial login, and divide it by the number of total players.

The result format is in the following example.

 

Example 1:

Input: \
Activity table:

| player_id | device_id | event_date | games_played |
|-----------|-----------|------------|--------------|
| 1         | 2         | 2016-03-01 | 5            |
| 1         | 2         | 2016-03-02 | 6            |
| 2         | 3         | 2017-06-25 | 1            |
| 3         | 1         | 2016-03-02 | 0            |
| 3         | 4         | 2018-07-03 | 5            |

Output: 

| fraction  |
|-----------|
| 0.33      |

Explanation: \
Only the player with id 1 logged back in after the first day he had logged in so the answer is 1/3 = 0.33

In [4]:
from datetime import timedelta

def sol_2(activity: pd.DataFrame) -> pd.DataFrame:
    dates = activity.groupby('player_id').agg({'event_date' : 'min'}).reset_index()
    dates['day_after_login'] = dates['event_date'] + timedelta(days=1)
    total = dates.shape[0]
    target = activity.merge(dates, left_on=['player_id', 'event_date'], right_on=['player_id', 'day_after_login'], how='inner')['player_id'].unique().shape[0]
    return pd.DataFrame({'fraction' : [round(target / total, 2)]})

In [5]:
def _single_value(df, col):
    if col not in df.columns or len(df) != 1:
        raise AssertionError(f"Expected 1-row DataFrame with column {col}, got shape={df.shape}, cols={list(df.columns)}")
    return float(df.iloc[0][col])


def _case(rows, expected_fraction_2dp):
    activity = pd.DataFrame(rows, columns=["player_id", "device_id", "event_date", "games_played"])
    activity["event_date"] = pd.to_datetime(activity["event_date"])
    got = sol_2(activity)
    val = _single_value(got, "fraction")
    if round(val + 1e-12, 2) != expected_fraction_2dp:
        raise AssertionError(f"Expected {expected_fraction_2dp:.2f}, got {val}")


def run():
    cases = [
        ("example", lambda: _case(
            [
                (1, 2, "2016-03-01", 5),
                (1, 2, "2016-03-02", 6),
                (2, 3, "2017-06-25", 1),
                (3, 1, "2016-03-02", 0),
                (3, 4, "2018-07-03", 5),
            ],
            0.33,
        )),
        ("all_return", lambda: _case(
            [
                (1, 1, "2020-01-01", 0), (1, 1, "2020-01-02", 0),
                (2, 1, "2020-02-10", 0), (2, 1, "2020-02-11", 0),
                (3, 1, "2020-03-05", 0), (3, 1, "2020-03-06", 0),
            ],
            1.00,
        )),
        ("none_return", lambda: _case(
            [
                (1, 1, "2020-01-01", 0),
                (2, 1, "2020-01-01", 0),
                (3, 1, "2020-01-01", 0),
                (1, 1, "2020-01-03", 0),  # not next day
            ],
            0.00,
        )),
        ("return_later_not_count", lambda: _case(
            [
                (1, 1, "2020-01-01", 0),
                (1, 1, "2020-01-04", 0),
                (2, 1, "2020-01-10", 0),
            ],
            0.00,
        )),
        ("first_login_has_multiple_entries", lambda: _case(
            [
                (1, 1, "2020-01-01", 1),
                (1, 2, "2020-01-01", 2),  # still first day
                (1, 1, "2020-01-02", 3),  # next day => counts
                (2, 1, "2020-01-05", 0),
            ],
            0.50,
        )),
        ("next_day_multiple_entries", lambda: _case(
            [
                (1, 1, "2020-01-01", 0),
                (1, 1, "2020-01-02", 0),
                (1, 2, "2020-01-02", 0),
                (2, 1, "2020-01-03", 0),
            ],
            0.50,
        )),
        ("single_player_no_next_day", lambda: _case(
            [(7, 1, "2019-12-31", 10)],
            0.00,
        )),
        ("single_player_has_next_day", lambda: _case(
            [(7, 1, "2019-12-31", 10), (7, 1, "2020-01-01", 0)],
            1.00,
        )),
        ("four_players_two_return", lambda: _case(
            [
                (1, 1, "2020-01-01", 0), (1, 1, "2020-01-02", 0),
                (2, 1, "2020-01-10", 0),
                (3, 1, "2020-02-01", 0), (3, 1, "2020-02-02", 0),
                (4, 1, "2020-03-01", 0), (4, 1, "2020-03-03", 0),
            ],
            0.50,
        )),
        ("rounding_check_1_of_6", lambda: _case(
            [
                (1, 1, "2020-01-01", 0), (1, 1, "2020-01-02", 0),
                (2, 1, "2020-01-01", 0),
                (3, 1, "2020-01-01", 0),
                (4, 1, "2020-01-01", 0),
                (5, 1, "2020-01-01", 0),
                (6, 1, "2020-01-01", 0),
            ],
            0.17,
        )),
    ]

    failed = 0
    for _, f in cases:
        try:
            f()
        except Exception:
            failed += 1
    print(failed)

run()

0


### Exercise 3

Table: `employee`


| Column Name | Type    |
|-------------|---------|
| id          | int     |
| name        | varchar |
| department  | varchar |
| managerId   | int     |

id is the primary key (column with unique values) for this table.\
Each row of this table indicates the name of an employee, their department, and the id of their manager.\
If managerId is null, then the employee does not have a manager.\
No employee will be the manager of themself.
 

Write a solution to find managers with at least five direct reports.

Return the result table in any order.

The result format is in the following example.

 

Example 1:

Input: \
Employee table:

| id  | name  | department | managerId |
|-----|-------|------------|-----------|
| 101 | John  | A          | null      |
| 102 | Dan   | A          | 101       |
| 103 | James | A          | 101       |
| 104 | Amy   | A          | 101       |
| 105 | Anne  | A          | 101       |
| 106 | Ron   | B          | 101       |

Output:
| name |
|------|
| John |


In [57]:
def sol_3(df: pd.DataFrame) -> pd.DataFrame:
    # print(df)
    # print(type(df.groupby('managerId')), type(df.groupby('managerId').count()), end='\n\n')
    # print(df.groupby('managerId').count())
    # print(f'\n{df.groupby('managerId').count().reset_index()}')
    # print('\n\n\n\n')

    
    df_c = df.copy()
    df = df.groupby('managerId').count().reset_index()
    # print(df)
    # print(df.loc[101, :])
    output = df[df['id'] >= 5]['managerId']
    return df_c[df_c['id'].isin(output)][['name']]

In [58]:
def _sort(df: pd.DataFrame) -> pd.DataFrame:
    if len(df.columns) == 0:
        return df
    return df.sort_values(list(df.columns)).reset_index(drop=True)


def _case(rows, expected_names):
    employee = pd.DataFrame(rows, columns=["id", "name", "department", "managerId"])
    employee["managerId"] = employee["managerId"].astype("Int64")
    got = sol_3(employee)
    exp = pd.DataFrame({"name": expected_names})
    assert_frame_equal(_sort(got), _sort(exp), check_dtype=False)


def run():
    cases = [
        ("example", lambda: _case(
            [
                (101, "John", "A", None),
                (102, "Dan", "A", 101),
                (103, "James", "A", 101),
                (104, "Amy", "A", 101),
                (105, "Anne", "A", 101),
                (106, "Ron", "B", 101),
            ],
            ["John"],
        )),
        ("none_qualify", lambda: _case(
            [(1, "A", "X", None), (2, "B", "X", 1), (3, "C", "X", 1), (4, "D", "X", 2), (5, "E", "X", 2)],
            [],
        )),
        ("exactly_five_reports", lambda: _case(
            [(1, "M", "X", None)] + [(i, f"e{i}", "X", 1) for i in range(2, 7)],
            ["M"],
        )),
        ("six_reports", lambda: _case(
            [(1, "M", "X", None)] + [(i, f"e{i}", "X", 1) for i in range(2, 8)],
            ["M"],
        )),
        ("two_managers_qualify", lambda: _case(
            [(1, "M1", "X", None), (2, "M2", "X", None)]
            + [(i, f"a{i}", "X", 1) for i in range(3, 8)]
            + [(i, f"b{i}", "X", 2) for i in range(8, 13)],
            ["M1", "M2"],
        )),
        ("reports_in_other_dept_still_count", lambda: _case(
            [(1, "Boss", "A", None)]
            + [(2, "x", "A", 1), (3, "y", "B", 1), (4, "z", "C", 1), (5, "u", "A", 1), (6, "v", "B", 1)],
            ["Boss"],
        )),
        ("manager_not_listed_as_employee_no_match", lambda: _case(
            [(1, "A", "X", 999), (2, "B", "X", 999), (3, "C", "X", 999), (4, "D", "X", 999), (5, "E", "X", 999)],
            [],  # no row for managerId=999 => cannot output a name
        )),
        ("many_employees_noise", lambda: _case(
            [(1, "M", "X", None)]
            + [(i, f"r{i}", "X", 1) for i in range(2, 7)]
            + [(7, "N", "X", None), (8, "a", "X", 7), (9, "b", "X", 7), (10, "c", "X", 7), (11, "d", "X", 7)],
            ["M"],  # N has only 4 reports
        )),
        ("manager_with_nulls", lambda: _case(
            [(1, "M", "X", None)]
            + [(2, "a", "X", 1), (3, "b", "X", 1), (4, "c", "X", 1), (5, "d", "X", 1), (6, "e", "X", 1)]
            + [(7, "solo", "X", None)],
            ["M"],
        )),
        ("id_large_values", lambda: _case(
            [(1000, "BigBoss", "X", None)]
            + [(i, f"e{i}", "X", 1000) for i in range(1001, 1006)],
            ["BigBoss"],
        )),
    ]

    failed = 0
    for _, f in cases:
        try:
            f()
        except Exception:
            failed += 1
    print(failed)

run()

0


### Exercise 4

Table: `insurance`


| Column Name | Type  |
|-------------|-------|
| pid         | int   |
| tiv_2015    | float |
| tiv_2016    | float |
| lat         | float |
| lon         | float |

pid is the primary key (column with unique values) for this table.\
Each row of this table contains information about one policy where:\
pid is the policyholder's policy ID.\
tiv_2015 is the total investment value in 2015 and tiv_2016 is the total investment value in 2016.\
lat is the latitude of the policy holder's city. It's guaranteed that lat is not NULL.\
lon is the longitude of the policy holder's city. It's guaranteed that lon is not NULL.
 

Write a solution to report the sum of all total investment values in 2016 tiv_2016, for all policyholders who:

have the same tiv_2015 value as one or more other policyholders, and
are not located in the same city as any other policyholder (i.e., the (lat, lon) attribute pairs must be unique).\
Round tiv_2016 to two decimal places.

The result format is in the following example.

 

Example 1:

Input: \
Insurance table:

| pid | tiv_2015 | tiv_2016 | lat | lon |
|-----|----------|----------|-----|-----|
| 1   | 10       | 5        | 10  | 10  |
| 2   | 20       | 20       | 20  | 20  |
| 3   | 10       | 30       | 20  | 20  |
| 4   | 10       | 40       | 40  | 40  |

Output: 

| tiv_2016 |
|----------|
| 45.00    |

Explanation: \
The first record in the table, like the last record, meets both of the two criteria.\
The tiv_2015 value 10 is the same as the third and fourth records, and its location is unique.

The second record does not meet any of the two criteria. Its tiv_2015 is not like any other policyholders and its location is the same as the third record, which makes the third record fail, too.\
So, the result is the sum of tiv_2016 of the first and last record, which is 45.

In [126]:
def sol_4(df: pd.DataFrame) -> pd.DataFrame:
    unique = df.groupby('tiv_2015').count()
    unique_tiv_2015 = unique[unique['lat'] > 1].index
    unique = df.groupby('lat').count()
    unique_lat = unique[unique['tiv_2016'] == 1].index
    
    unique = df.groupby('lon').count()
    unique_lon = unique[unique['tiv_2016'] == 1].index

    res = df[(df['tiv_2015'].isin(unique_tiv_2015)) & ( (df['lat'].isin(unique_lat)) | (df['lon'].isin(unique_lon)) )]
    # print(f'{df}\n\n{res}\n=========')
    # print(res['tiv_2016'].sum())
    res = pd.DataFrame(data=[res['tiv_2016'].sum()], columns=['tiv_2016'])
    res['tiv_2016'] = res['tiv_2016'].astype(float).round(2)
    # print(res)
    return res

In [127]:
def _single_value(df, col):
    if col not in df.columns or len(df) != 1:
        raise AssertionError(f"Expected 1-row DataFrame with column {col}")
    return float(df.iloc[0][col])


def _case(rows, expected_sum_2dp):
    ins = pd.DataFrame(rows, columns=["pid", "tiv_2015", "tiv_2016", "lat", "lon"])
    got = sol_4(ins)
    val = _single_value(got, "tiv_2016")
    if round(val + 1e-12, 2) != expected_sum_2dp:
        raise AssertionError(f"Expected {expected_sum_2dp:.2f}, got {val}")


def run():
    cases = [
        ("example", lambda: _case(
            [(1, 10, 5, 10, 10), (2, 20, 20, 20, 20), (3, 10, 30, 20, 20), (4, 10, 40, 40, 40)],
            45.00,
        )),
        ("no_duplicate_tiv2015", lambda: _case(
            [(1, 10, 5, 1, 1), (2, 20, 7, 2, 2)],
            0.00,
        )),
        ("duplicate_tiv2015_but_all_locations_duplicate", lambda: _case(
            [(1, 10, 5, 1, 1), (2, 10, 7, 1, 1)],
            0.00,
        )),
        ("one_location_duplicate_kills_both", lambda: _case(
            [(1, 10, 5, 1, 1), (2, 10, 7, 1, 1), (3, 10, 11, 2, 2)],
            11.00,  # only pid=3 has unique location and tiv2015 duplicated
        )),
        ("multiple_tiv_groups", lambda: _case(
            [
                (1, 10, 1.5, 1, 1),
                (2, 10, 2.5, 2, 2),
                (3, 20, 3.0, 3, 3),
                (4, 20, 4.0, 4, 4),
                (5, 30, 5.0, 4, 4),  # location duplicate => disqualify pid 4 and 5, and tiv2015=30 not duplicated
            ],
            7.00,  # pid 1+2+3? wait: tiv2015=20 duplicated (3,4) but pid4 disqualified by location dup, pid3 ok => include 3.0; tiv2015=10 duplicated (1,2) include 1.5+2.5
        )),
        ("decimal_rounding", lambda: _case(
            [(1, 10, 1.005, 1, 1), (2, 10, 2.005, 2, 2)],
            3.01,  # 1.005+2.005=3.01
        )),
        ("negative_tiv2016_allowed", lambda: _case(
            [(1, 10, -5, 1, 1), (2, 10, 1, 2, 2)],
            -4.00,
        )),
        ("larger_set_mixed", lambda: _case(
            [
                (1, 10, 10, 10, 10),
                (2, 10, 20, 20, 20),
                (3, 10, 30, 20, 20),  # location dup with pid2 => both pid2/pid3 disqualify
                (4, 20, 40, 30, 30),
                (5, 20, 50, 40, 40),
                (6, 30, 60, 50, 50),  # tiv2015 not dup => exclude
            ],
            10.00 + 40.00 + 50.00,  # pid1 qualifies; pid4+pid5 qualify
        )),
        ("single_row", lambda: _case(
            [(1, 10, 5, 1, 1)],
            0.00,
        )),
        ("duplicate_tiv_many_unique_locations", lambda: _case(
            [(i, 99, float(i), float(i), float(i)) for i in range(1, 6)],
            15.00,  # sum 1+2+3+4+5
        )),
    ]

    failed = 0
    for _, f in cases:
        # try:
        f()
        # except Exception:
            # failed += 1
    print(failed)

run()

0


### Exercise 5

Table: `RequestAccepted`


| Column Name    | Type    |
|----------------|---------|
| requester_id   | int     |
| accepter_id    | int     |
| accept_date    | date    |

(requester_id, accepter_id) is the primary key (combination of columns with unique values) for this table.\
This table contains the ID of the user who sent the request, the ID of the user who received the request, and the date when the request was accepted.
 

Write a solution to find the people who have the most friends and the most friends number.

The test cases are generated so that only one person has the most friends.

The result format is in the following example.

 

Example 1:

Input: \
RequestAccepted table:

| requester_id | accepter_id | accept_date |
|--------------|-------------|-------------|
| 1            | 2           | 2016/06/03  |
| 1            | 3           | 2016/06/08  |
| 2            | 3           | 2016/06/08  |
| 3            | 4           | 2016/06/09  |

Output: 

| id | num |
|----|-----|
| 3  | 3   |

Explanation:\
The person with id 3 is a friend of people 1, 2, and 4, so he has three friends in total, which is the most number than any others.

In [ ]:
###your solution here

In [ ]:
def _norm(df: pd.DataFrame) -> pd.DataFrame:
    return df.sort_values(list(df.columns)).reset_index(drop=True)


def _case(rows, expected_id, expected_num):
    ra = pd.DataFrame(rows, columns=["requesterid", "accepterid", "acceptdate"])
    ra["acceptdate"] = pd.to_datetime(ra["acceptdate"])
    got = sol_5(ra)
    exp = pd.DataFrame({"id": [expected_id], "num": [expected_num]})
    assert_frame_equal(_norm(got), _norm(exp), check_dtype=False)


def run():
    cases = [
        ("example", lambda: _case(
            [(1, 2, "2016-06-03"), (1, 3, "2016-06-08"), (2, 3, "2016-06-08"), (3, 4, "2016-06-09")],
            3, 3,
        )),
        ("star_center_requester", lambda: _case(
            [(10, 1, "2020-01-01"), (10, 2, "2020-01-02"), (10, 3, "2020-01-03"), (10, 4, "2020-01-04")],
            10, 4,
        )),
        ("star_center_accepter", lambda: _case(
            [(1, 10, "2020-01-01"), (2, 10, "2020-01-02"), (3, 10, "2020-01-03")],
            10, 3,
        )),
        ("chain_graph", lambda: _case(
            [(1, 2, "2020-01-01"), (2, 3, "2020-01-02"), (3, 4, "2020-01-03")],
            2, 2,  # degrees: 1->1,2->2,3->2,4->1, tie exists but prompt says unique in real tests; here pick 2 to enforce stable? might fail if tie-handling differs
        )),
        ("triangle_unique_top_added", lambda: _case(
            [(1, 2, "2020-01-01"), (2, 3, "2020-01-02"), (1, 3, "2020-01-03"), (3, 4, "2020-01-04")],
            3, 3,
        )),
        ("two_components", lambda: _case(
            [(1, 2, "2020-01-01"), (2, 3, "2020-01-02"), (10, 11, "2020-01-03"), (10, 12, "2020-01-04"), (10, 13, "2020-01-05")],
            10, 3,
        )),
        ("dates_irrelevant", lambda: _case(
            [(1, 2, "2010-01-01"), (1, 3, "1999-12-31"), (4, 1, "2050-05-05")],
            1, 3,
        )),
        ("bigger_star", lambda: _case(
            [(100, i, "2021-01-01") for i in range(1, 8)],
            100, 7,
        )),
        ("repeated_node_many_edges", lambda: _case(
            [(1, 2, "2020-01-01"), (1, 3, "2020-01-01"), (1, 4, "2020-01-01"), (5, 1, "2020-01-01"), (6, 1, "2020-01-01")],
            1, 5,
        )),
        ("minimal_two_users", lambda: _case(
            [(1, 2, "2020-01-01")],
            1, 1,  # tie again; kept as simple sanity
        )),
    ]

    # NOTE: If your sol_5 explicitly assumes the "unique top" condition, you may want to remove/adjust tie-cases above.

    failed = 0
    for _, f in cases:
        try:
            f()
        except Exception:
            failed += 1
    print(failed)

run()

### Exercise 6

Table: `Seat`

| Column Name | Type    |
|-------------|---------|
| id          | int     |
| student     | varchar |

id is the primary key (unique value) column for this table.\
Each row of this table indicates the name and the ID of a student.\
The ID sequence always starts from 1 and increments continuously.
 

Write a solution to swap the seat id of every two consecutive students. If the number of students is odd, the id of the last student is not swapped.

Return the result table ordered by id in ascending order.

The result format is in the following example.

 

Example 1:

Input: \
Seat table:

| id | student |
|----|---------|
| 1  | Abbot   |
| 2  | Doris   |
| 3  | Emerson |
| 4  | Green   |
| 5  | Jeames  |

Output: 

| id | student |
|----|---------|
| 1  | Doris   |
| 2  | Abbot   |
| 3  | Green   |
| 4  | Emerson |
| 5  | Jeames  |

Explanation: \
Note that if the number of students is odd, there is no need to change the last one's seat.

In [ ]:
###your solution here

In [ ]:
def _norm(df: pd.DataFrame) -> pd.DataFrame:
    return df.sort_values("id").reset_index(drop=True)


def _case(rows, exp_rows):
    seat = pd.DataFrame(rows, columns=["id", "student"])
    got = sol_6(seat)
    exp = pd.DataFrame(exp_rows, columns=["id", "student"])
    assert_frame_equal(_norm(got), _norm(exp), check_dtype=False)


def run():
    cases = [
        ("example", lambda: _case(
            [(1, "Abbot"), (2, "Doris"), (3, "Emerson"), (4, "Green"), (5, "Jeames")],
            [(1, "Doris"), (2, "Abbot"), (3, "Green"), (4, "Emerson"), (5, "Jeames")],
        )),
        ("even_4", lambda: _case(
            [(1, "a"), (2, "b"), (3, "c"), (4, "d")],
            [(1, "b"), (2, "a"), (3, "d"), (4, "c")],
        )),
        ("single_row", lambda: _case(
            [(1, "solo")],
            [(1, "solo")],
        )),
        ("two_rows", lambda: _case(
            [(1, "x"), (2, "y")],
            [(1, "y"), (2, "x")],
        )),
        ("three_rows", lambda: _case(
            [(1, "x"), (2, "y"), (3, "z")],
            [(1, "y"), (2, "x"), (3, "z")],
        )),
        ("six_rows", lambda: _case(
            [(1, "s1"), (2, "s2"), (3, "s3"), (4, "s4"), (5, "s5"), (6, "s6")],
            [(1, "s2"), (2, "s1"), (3, "s4"), (4, "s3"), (5, "s6"), (6, "s5")],
        )),
        ("names_repeat", lambda: _case(
            [(1, "a"), (2, "a"), (3, "b"), (4, "b"), (5, "c")],
            [(1, "a"), (2, "a"), (3, "b"), (4, "b"), (5, "c")],  # swapping identicals no visible change except positions
        )),
        ("non_string_students", lambda: _case(
            [(1, 10), (2, 20), (3, 30), (4, 40), (5, 50)],
            [(1, 20), (2, 10), (3, 40), (4, 30), (5, 50)],
        )),
        ("longer_odd", lambda: _case(
            [(i, f"u{i}") for i in range(1, 8)],
            [(1, "u2"), (2, "u1"), (3, "u4"), (4, "u3"), (5, "u6"), (6, "u5"), (7, "u7")],
        )),
        ("already_swapped_input_should_swap_again", lambda: _case(
            [(1, "B"), (2, "A"), (3, "D"), (4, "C")],
            [(1, "A"), (2, "B"), (3, "C"), (4, "D")],
        )),
    ]

    failed = 0
    for _, f in cases:
        try:
            f()
        except Exception:
            failed += 1
    print(failed)


run()


### Exercise 7

Table: `Customer`


| Column Name | Type    |
|-------------|---------|
| customer_id | int     |
| product_key | int     |

This table may contain duplicates rows. \
customer_id is not NULL.\
product_key is a foreign key (reference column) to Product table.
 

Table: Product


| Column Name | Type    |
|-------------|---------|
| product_key | int     |

product_key is the primary key (column with unique values) for this table.
 

Write a solution to report the customer ids from the Customer table that bought all the products in the Product table.

Return the result table in any order.

The result format is in the following example.

 

Example 1:

Input: \
Customer table:

| customer_id | product_key |
|-------------|-------------|
| 1           | 5           |
| 2           | 6           |
| 3           | 5           |
| 3           | 6           |
| 1           | 6           |

Product table:

| product_key |
|-------------|
| 5           |
| 6           |

Output: 
| customer_id |
|-------------|
| 1           |
| 3           |

Explanation:\
The customers who bought all the products (5 and 6) are customers with IDs 1 and 3.

In [ ]:
###your solution here

In [ ]:
def _norm(df: pd.DataFrame) -> pd.DataFrame:
    return df.sort_values(["customerid"]).reset_index(drop=True)


def _case(customer_rows, product_rows, expected_customers):
    customer = pd.DataFrame(customer_rows, columns=["customerid", "productkey"])
    product = pd.DataFrame(product_rows, columns=["productkey"])
    got = sol_7(customer, product)
    exp = pd.DataFrame({"customerid": expected_customers})
    assert_frame_equal(_norm(got), _norm(exp), check_dtype=False)


def run():
    cases = [
        ("example", lambda: _case(
            [(1, 5), (2, 6), (3, 5), (3, 6), (1, 6)],
            [(5,), (6,)],
            [1, 3],
        )),
        ("single_product_everyone_who_bought_it", lambda: _case(
            [(1, 9), (2, 9), (3, 8)],
            [(9,)],
            [1, 2],
        )),
        ("duplicates_in_customer", lambda: _case(
            [(1, 1), (1, 1), (1, 2), (2, 1), (2, 2), (2, 2)],
            [(1,), (2,)],
            [1, 2],
        )),
        ("extra_products_in_customer_ignored", lambda: _case(
            [(1, 1), (1, 2), (1, 999), (2, 1)],
            [(1,), (2,)],
            [1],
        )),
        ("no_one_buys_all", lambda: _case(
            [(1, 1), (2, 2), (3, 3)],
            [(1,), (2,), (3,), (4,)],
            [],
        )),
        ("two_winners", lambda: _case(
            [(1, 1), (1, 2), (2, 1), (2, 2), (3, 1)],
            [(1,), (2,)],
            [1, 2],
        )),
        ("product_table_has_many", lambda: _case(
            [(1, 1), (1, 2), (1, 3), (2, 1), (2, 2), (3, 1), (3, 2), (3, 3)],
            [(1,), (2,), (3,)],
            [1, 3],
        )),
        ("customer_has_nulls_should_not_help", lambda: _case(
            [(1, 1), (1, None), (2, 1), (2, 2)],
            [(1,), (2,)],
            [2],
        )),
        ("product_keys_unsorted", lambda: _case(
            [(1, 2), (1, 1), (2, 1)],
            [(2,), (1,)],
            [1],
        )),
        ("large_ids", lambda: _case(
            [(100, 5), (100, 6), (200, 5)],
            [(5,), (6,)],
            [100],
        )),
    ]

    failed = 0
    for _, f in cases:
        try:
            f()
        except Exception:
            failed += 1
    print(failed)


run()

### Exercise 8

Table: `Users`


| Column Name    | Type    |
|----------------|---------|
| user_id        | int     |
| join_date      | date    |
| favorite_brand | varchar |

user_id is the primary key (column with unique values) of this table.\
This table has the info of the users of an online shopping website where users can sell and buy items.
 

Table: Orders


| Column Name   | Type    |
|---------------|---------|
| order_id      | int     |
| order_date    | date    |
| item_id       | int     |
| buyer_id      | int     |
| seller_id     | int     |

order_id is the primary key (column with unique values) of this table.\
item_id is a foreign key (reference column) to the Items table.\
buyer_id and seller_id are foreign keys to the Users table.
 

Table: Items


| Column Name   | Type    |
|---------------|---------|
| item_id       | int     |
| item_brand    | varchar |

item_id is the primary key (column with unique values) of this table.
 

Write a solution to find for each user, the join date and the number of orders they made as a buyer in 2019.

Return the result table in any order.

The result format is in the following example.

 

Example 1:

Input: \
Users table:

| user_id | join_date  | favorite_brand |
|---------|------------|----------------|
| 1       | 2018-01-01 | Lenovo         |
| 2       | 2018-02-09 | Samsung        |
| 3       | 2018-01-19 | LG             |
| 4       | 2018-05-21 | HP             |

Orders table:

| order_id | order_date | item_id | buyer_id | seller_id |
|----------|------------|---------|----------|-----------|
| 1        | 2019-08-01 | 4       | 1        | 2         |
| 2        | 2018-08-02 | 2       | 1        | 3         |
| 3        | 2019-08-03 | 3       | 2        | 3         |
| 4        | 2018-08-04 | 1       | 4        | 2         |
| 5        | 2018-08-04 | 1       | 3        | 4         |
| 6        | 2019-08-05 | 2       | 2        | 4         |

Items table:

| item_id | item_brand |
|---------|------------|
| 1       | Samsung    |
| 2       | Lenovo     |
| 3       | LG         |
| 4       | HP         |

Output: 

| buyer_id  | join_date  | orders_in_2019 |
|-----------|------------|----------------|
| 1         | 2018-01-01 | 1              |
| 2         | 2018-02-09 | 2              |
| 3         | 2018-01-19 | 0              |
| 4         | 2018-05-21 | 0              |


In [ ]:
###your solution here

In [ ]:
def _norm(df: pd.DataFrame) -> pd.DataFrame:
    return df.sort_values(["buyerid"]).reset_index(drop=True)


def _case(users_rows, orders_rows, expected_rows):
    users = pd.DataFrame(users_rows, columns=["userid", "joindate", "favoritebrand"])
    users["joindate"] = pd.to_datetime(users["joindate"])
    orders = pd.DataFrame(orders_rows, columns=["orderid", "orderdate", "itemid", "buyerid", "sellerid"])
    orders["orderdate"] = pd.to_datetime(orders["orderdate"])
    got = sol_8(users, orders)
    exp = pd.DataFrame(expected_rows, columns=["buyerid", "joindate", "ordersin2019"])
    exp["joindate"] = pd.to_datetime(exp["joindate"])
    assert_frame_equal(_norm(got), _norm(exp), check_dtype=False)


def run():
    cases = [
        ("example", lambda: _case(
            [
                (1, "2018-01-01", "Lenovo"),
                (2, "2018-02-09", "Samsung"),
                (3, "2018-01-19", "LG"),
                (4, "2018-05-21", "HP"),
            ],
            [
                (1, "2019-08-01", 4, 1, 2),
                (2, "2018-08-02", 2, 1, 3),
                (3, "2019-08-03", 3, 2, 3),
                (4, "2018-08-04", 1, 4, 2),
                (5, "2018-08-04", 1, 3, 4),
                (6, "2019-08-05", 2, 2, 4),
            ],
            [
                (1, "2018-01-01", 1),
                (2, "2018-02-09", 2),
                (3, "2018-01-19", 0),
                (4, "2018-05-21", 0),
            ],
        )),
        ("no_orders_2019", lambda: _case(
            [(1, "2017-01-01", "X")],
            [(1, "2018-12-31", 1, 1, 2)],
            [(1, "2017-01-01", 0)],
        )),
        ("orders_only_2019", lambda: _case(
            [(1, "2017-01-01", "X"), (2, "2017-01-02", "Y")],
            [(1, "2019-01-01", 1, 1, 2), (2, "2019-12-31", 2, 1, 2), (3, "2019-06-01", 3, 2, 1)],
            [(1, "2017-01-01", 2), (2, "2017-01-02", 1)],
        )),
        ("orders_in_2020_ignored", lambda: _case(
            [(1, "2017-01-01", "X")],
            [(1, "2020-01-01", 1, 1, 2), (2, "2019-12-31", 1, 1, 2)],
            [(1, "2017-01-01", 1)],
        )),
        ("multiple_orders_same_day", lambda: _case(
            [(1, "2018-01-01", "A")],
            [(1, "2019-05-05", 1, 1, 2), (2, "2019-05-05", 2, 1, 3), (3, "2019-05-05", 3, 1, 4)],
            [(1, "2018-01-01", 3)],
        )),
        ("buyerid_not_in_users_should_not_appear", lambda: _case(
            [(1, "2018-01-01", "A")],
            [(1, "2019-01-01", 1, 999, 1)],
            [(1, "2018-01-01", 0)],
        )),
        ("seller_is_user_does_not_count", lambda: _case(
            [(1, "2018-01-01", "A"), (2, "2018-01-02", "B")],
            [(1, "2019-01-01", 1, 2, 1)],  # user1 sold, user2 bought
            [(1, "2018-01-01", 0), (2, "2018-01-02", 1)],
        )),
        ("empty_orders", lambda: _case(
            [(1, "2018-01-01", "A"), (2, "2018-01-02", "B")],
            [],
            [(1, "2018-01-01", 0), (2, "2018-01-02", 0)],
        )),
        ("orders_spanning_year_boundary", lambda: _case(
            [(1, "2018-01-01", "A")],
            [(1, "2018-12-31", 1, 1, 2), (2, "2019-01-01", 1, 1, 2), (3, "2019-12-31", 1, 1, 2), (4, "2020-01-01", 1, 1, 2)],
            [(1, "2018-01-01", 2)],
        )),
        ("many_users_sparse_orders", lambda: _case(
            [(i, "2018-01-01", "X") for i in range(1, 6)],
            [(1, "2019-02-02", 1, 1, 2), (2, "2019-03-03", 1, 5, 4)],
            [(1, "2018-01-01", 1), (2, "2018-01-01", 0), (3, "2018-01-01", 0), (4, "2018-01-01", 0), (5, "2018-01-01", 1)],
        )),
    ]

    failed = 0
    for _, f in cases:
        try:
            f()
        except Exception:
            failed += 1
    print(failed)

run()

### Exercise 9

Table: `Products`


| Column Name   | Type    |
|---------------|---------|
| product_id    | int     |
| new_price     | int     |
| change_date   | date    |

(product_id, change_date) is the primary key (combination of columns with unique values) of this table.\
Each row of this table indicates that the price of some product was changed to a new price at some date.\
Initially, all products have price 10.

Write a solution to find the prices of all products on the date 2019-08-16.

Return the result table in any order.

The result format is in the following example.

 

Example 1:

Input: \
Products table:

| product_id | new_price | change_date |
|------------|-----------|-------------|
| 1          | 20        | 2019-08-14  |
| 2          | 50        | 2019-08-14  |
| 1          | 30        | 2019-08-15  |
| 1          | 35        | 2019-08-16  |
| 2          | 65        | 2019-08-17  |
| 3          | 20        | 2019-08-18  |

Output: 

| product_id | price |
|------------|-------|
| 2          | 50    |
| 1          | 35    |
| 3          | 10    |


In [ ]:
###your solution here

In [ ]:
def _norm(df: pd.DataFrame) -> pd.DataFrame:
    return df.sort_values(["productid"]).reset_index(drop=True)


def _case(rows, expected_rows):
    products = pd.DataFrame(rows, columns=["productid", "newprice", "changedate"])
    products["changedate"] = pd.to_datetime(products["changedate"])
    got = sol_9(products)
    exp = pd.DataFrame(expected_rows, columns=["productid", "price"])
    assert_frame_equal(_norm(got), _norm(exp), check_dtype=False)


def run():
    cases = [
        ("example", lambda: _case(
            [(1, 20, "2019-08-14"), (2, 50, "2019-08-14"), (1, 30, "2019-08-15"), (1, 35, "2019-08-16"), (2, 65, "2019-08-17"), (3, 20, "2019-08-18")],
            [(1, 35), (2, 50), (3, 10)],
        )),
        ("only_after_target", lambda: _case(
            [(1, 99, "2019-08-17"), (1, 5, "2019-08-18")],
            [(1, 10)],
        )),
        ("change_before_only", lambda: _case(
            [(1, 15, "2019-08-01")],
            [(1, 15)],
        )),
        ("multiple_changes_before_target_take_latest", lambda: _case(
            [(1, 11, "2019-08-01"), (1, 12, "2019-08-10"), (1, 13, "2019-08-15")],
            [(1, 13)],
        )),
        ("change_on_target_date_counts", lambda: _case(
            [(1, 22, "2019-08-16")],
            [(1, 22)],
        )),
        ("multiple_products_some_missing", lambda: _case(
            [(1, 20, "2019-08-14"), (2, 50, "2019-08-14"), (4, 7, "2019-08-10")],
            [(1, 20), (2, 50), (4, 7)],
        )),
        ("product_with_change_after_target_keeps_old", lambda: _case(
            [(2, 50, "2019-08-14"), (2, 65, "2019-08-17")],
            [(2, 50)],
        )),
        ("unsorted_input_dates", lambda: _case(
            [(1, 30, "2019-08-15"), (1, 20, "2019-08-14"), (1, 35, "2019-08-16")],
            [(1, 35)],
        )),
        ("several_products_no_changes_any", lambda: _case(
            [],  # if your sol_9 expects Products table always non-empty, adjust/remove
            [],
        )),
        ("prices_ints_return", lambda: _case(
            [(3, 99, "2019-08-15"), (3, 100, "2019-08-16")],
            [(3, 100)],
        )),
    ]

    failed = 0
    for _, f in cases:
        try:
            f()
        except Exception:
            failed += 1
    print(failed)

run()

### Exercise 10

Table: `Delivery`


| Column Name                 | Type    |
|-----------------------------|---------|
| delivery_id                 | int     |
| customer_id                 | int     |
| order_date                  | date    |
| customer_pref_delivery_date | date    |
+-----------------------------+---------+
delivery_id is the column of unique values of this table.\
The table holds information about food delivery to customers that make orders at some date and specify a preferred delivery date (on the same order date or after it).
 

If the customer's preferred delivery date is the same as the order date, then the order is called immediate; otherwise, it is called scheduled.

The first order of a customer is the order with the earliest order date that the customer made. It is guaranteed that a customer has precisely one first order.

Write a solution to find the percentage of immediate orders in the first orders of all customers, rounded to 2 decimal places.

The result format is in the following example.

 

Example 1:

Input: \
Delivery table:

| delivery_id | customer_id | order_date | customer_pref_delivery_date |
|-------------|-------------|------------|-----------------------------|
| 1           | 1           | 2019-08-01 | 2019-08-02                  |
| 2           | 2           | 2019-08-02 | 2019-08-02                  |
| 3           | 1           | 2019-08-11 | 2019-08-12                  |
| 4           | 3           | 2019-08-24 | 2019-08-24                  |
| 5           | 3           | 2019-08-21 | 2019-08-22                  |
| 6           | 2           | 2019-08-11 | 2019-08-13                  |
| 7           | 4           | 2019-08-09 | 2019-08-09                  |

Output: 

| immediate_percentage |
|----------------------|
| 50.00                |

Explanation: \
The customer id 1 has a first order with delivery id 1 and it is scheduled.\
The customer id 2 has a first order with delivery id 2 and it is immediate.\
The customer id 3 has a first order with delivery id 5 and it is scheduled.\
The customer id 4 has a first order with delivery id 7 and it is immediate.\
Hence, half the customers have immediate first orders.

In [ ]:
###your solution here

In [ ]:
def _single_value(df, col):
    if col not in df.columns or len(df) != 1:
        raise AssertionError(f"Expected 1-row DataFrame with column {col}")
    return float(df.iloc[0][col])


def _case(rows, expected_pct_2dp):
    d = pd.DataFrame(rows, columns=["deliveryid", "customerid", "orderdate", "customerprefdeliverydate"])
    d["orderdate"] = pd.to_datetime(d["orderdate"])
    d["customerprefdeliverydate"] = pd.to_datetime(d["customerprefdeliverydate"])
    got = sol_10(d)
    val = _single_value(got, "immediatepercentage")
    if round(val + 1e-12, 2) != expected_pct_2dp:
        raise AssertionError(f"Expected {expected_pct_2dp:.2f}, got {val}")


def run():
    cases = [
        ("example", lambda: _case(
            [
                (1, 1, "2019-08-01", "2019-08-02"),
                (2, 2, "2019-08-02", "2019-08-02"),
                (3, 1, "2019-08-11", "2019-08-12"),
                (4, 3, "2019-08-24", "2019-08-24"),
                (5, 3, "2019-08-21", "2019-08-22"),
                (6, 2, "2019-08-11", "2019-08-13"),
                (7, 4, "2019-08-09", "2019-08-09"),
            ],
            50.00,
        )),
        ("all_immediate_first", lambda: _case(
            [(1, 1, "2020-01-01", "2020-01-01"), (2, 2, "2020-01-05", "2020-01-05")],
            100.00,
        )),
        ("none_immediate_first", lambda: _case(
            [(1, 1, "2020-01-01", "2020-01-02"), (2, 2, "2020-01-05", "2020-01-06")],
            0.00,
        )),
        ("later_orders_do_not_matter", lambda: _case(
            [(1, 1, "2020-01-01", "2020-01-02"), (2, 1, "2020-01-02", "2020-01-02")],
            0.00,
        )),
        ("unsorted_input_first_order_by_date", lambda: _case(
            [(2, 1, "2020-01-02", "2020-01-02"), (1, 1, "2020-01-01", "2020-01-03")],
            0.00,
        )),
        ("four_customers_two_immediate", lambda: _case(
            [(1, 1, "2020-01-01", "2020-01-01"), (2, 2, "2020-01-01", "2020-01-02"), (3, 3, "2020-01-02", "2020-01-02"), (4, 4, "2020-01-03", "2020-01-04")],
            50.00,
        )),
        ("customer_first_order_has_same_day_pref", lambda: _case(
            [(1, 1, "2020-05-01", "2020-05-01"), (2, 1, "2020-05-02", "2020-05-03")],
            100.00,
        )),
        ("fraction_1_of_3_rounding", lambda: _case(
            [(1, 1, "2020-01-01", "2020-01-02"), (2, 2, "2020-01-01", "2020-01-02"), (3, 3, "2020-01-01", "2020-01-01")],
            33.33,
        )),
        ("many_orders_many_customers", lambda: _case(
            [
                (1, 1, "2020-01-01", "2020-01-01"),
                (2, 1, "2020-01-10", "2020-01-11"),
                (3, 2, "2020-02-01", "2020-02-02"),
                (4, 2, "2020-02-03", "2020-02-03"),
                (5, 3, "2020-03-01", "2020-03-01"),
                (6, 4, "2020-04-01", "2020-04-02"),
            ],
            50.00,  # customers:1 immediate;2 scheduled (first is 02-01);3 immediate;4 scheduled => 2/4
        )),
        ("single_customer_single_order", lambda: _case(
            [(1, 9, "2020-01-01", "2020-01-01")],
            100.00,
        )),
    ]

    failed = 0
    for _, f in cases:
        try:
            f()
        except Exception:
            failed += 1
    print(failed)

run()

### Exercise 11

Table: `Transactions`


| Column Name   | Type    |
|---------------|---------|
| id            | int     |
| country       | varchar |
| state         | enum    |
| amount        | int     |
| trans_date    | date    |
+---------------+---------+
id is the primary key of this table.\
The table has information about incoming transactions.\
The state column is an enum of type ["approved", "declined"].
 

Write an SQL query to find for each month and country, the number of transactions and their total amount, the number of approved transactions and their total amount.

Return the result table in any order.

The query result format is in the following example.

 

Example 1:

Input: \
Transactions table:

| id   | country | state    | amount | trans_date |
|------|---------|----------|--------|------------|
| 121  | US      | approved | 1000   | 2018-12-18 |
| 122  | US      | declined | 2000   | 2018-12-19 |
| 123  | US      | approved | 2000   | 2019-01-01 |
| 124  | DE      | approved | 2000   | 2019-01-07 |

Output: 

| month    | country | trans_count | approved_count | trans_total_amount | approved_total_amount |
|----------|---------|-------------|----------------|--------------------|-----------------------|
| 2018-12  | US      | 2           | 1              | 3000               | 1000                  |
| 2019-01  | US      | 1           | 1              | 2000               | 2000                  |
| 2019-01  | DE      | 1           | 1              | 2000               | 2000                  |


In [ ]:
###your solution here

In [ ]:
def _norm(df: pd.DataFrame) -> pd.DataFrame:
    return df.sort_values(["month", "country"]).reset_index(drop=True)


def _case(rows, expected_rows):
    tx = pd.DataFrame(rows, columns=["id", "country", "state", "amount", "transdate"])
    tx["transdate"] = pd.to_datetime(tx["transdate"])
    got = sol_11(tx)
    exp = pd.DataFrame(
        expected_rows,
        columns=["month", "country", "transcount", "approvedcount", "transtotalamount", "approvedtotalamount"],
    )
    assert_frame_equal(_norm(got), _norm(exp), check_dtype=False)


def run():
    cases = [
        ("example", lambda: _case(
            [(121, "US", "approved", 1000, "2018-12-18"), (122, "US", "declined", 2000, "2018-12-19"), (123, "US", "approved", 2000, "2019-01-01"), (124, "DE", "approved", 2000, "2019-01-07")],
            [("2018-12", "US", 2, 1, 3000, 1000), ("2019-01", "DE", 1, 1, 2000, 2000), ("2019-01", "US", 1, 1, 2000, 2000)],
        )),
        ("single_row_approved", lambda: _case(
            [(1, "FI", "approved", 10, "2020-02-02")],
            [("2020-02", "FI", 1, 1, 10, 10)],
        )),
        ("single_row_declined", lambda: _case(
            [(1, "FI", "declined", 10, "2020-02-02")],
            [("2020-02", "FI", 1, 0, 10, 0)],
        )),
        ("two_months_same_country", lambda: _case(
            [(1, "US", "approved", 5, "2020-01-01"), (2, "US", "approved", 7, "2020-02-01")],
            [("2020-01", "US", 1, 1, 5, 5), ("2020-02", "US", 1, 1, 7, 7)],
        )),
        ("two_countries_same_month", lambda: _case(
            [(1, "US", "approved", 5, "2020-01-01"), (2, "DE", "declined", 7, "2020-01-02")],
            [("2020-01", "DE", 1, 0, 7, 0), ("2020-01", "US", 1, 1, 5, 5)],
        )),
        ("multiple_approved_declined", lambda: _case(
            [(1, "LV", "declined", 5, "2020-02-01"), (2, "LV", "approved", 7, "2020-02-02"), (3, "LV", "approved", 11, "2020-02-20")],
            [("2020-02", "LV", 3, 2, 23, 18)],
        )),
        ("amounts_sum_correct", lambda: _case(
            [(1, "US", "approved", 1, "2020-03-01"), (2, "US", "declined", 2, "2020-03-02"), (3, "US", "declined", 3, "2020-03-03")],
            [("2020-03", "US", 3, 1, 6, 1)],
        )),
        ("out_of_order_dates", lambda: _case(
            [(2, "US", "approved", 10, "2020-01-15"), (1, "US", "approved", 5, "2020-01-01")],
            [("2020-01", "US", 2, 2, 15, 15)],
        )),
        ("country_case_sensitive", lambda: _case(
            [(1, "us", "approved", 5, "2020-01-01"), (2, "US", "approved", 7, "2020-01-01")],
            [("2020-01", "US", 1, 1, 7, 7), ("2020-01", "us", 1, 1, 5, 5)],
        )),
        ("empty_input", lambda: _case(
            [],
            [],
        )),
    ]

    failed = 0
    for _, f in cases:
        try:
            f()
        except Exception:
            failed += 1
    print(failed)


run()

### Exercise 12

Table: `Queue`


| Column Name | Type    |
|-------------|---------|
| person_id   | int     |
| person_name | varchar |
| weight      | int     |
| turn        | int     |

person_id column contains unique values.\
This table has the information about all people waiting for a bus.\
The person_id and turn columns will contain all numbers from 1 to n, where n is the number of rows in the table.\
turn determines the order of which the people will board the bus, where turn=1 denotes the first person to board and turn=n denotes the last person to board.\
weight is the weight of the person in kilograms.
 

There is a queue of people waiting to board a bus. However, the bus has a weight limit of 1000 kilograms, so there may be some people who cannot board.

Write a solution to find the person_name of the last person that can fit on the bus without exceeding the weight limit. The test cases are generated such that the first person does not exceed the weight limit.

Note that only one person can board the bus at any given turn.

The result format is in the following example.

 

Example 1:

Input:\
Queue table:

| person_id | person_name | weight | turn |
|-----------|-------------|--------|------|
| 5         | Alice       | 250    | 1    |
| 4         | Bob         | 175    | 5    |
| 3         | Alex        | 350    | 2    |
| 6         | John Cena   | 400    | 3    |
| 1         | Winston     | 500    | 6    |
| 2         | Marie       | 200    | 4    |

Output: 

| person_name |
|-------------|
| John Cena   |

Explanation: The folowing table is ordered by the turn for simplicity.

| Turn | ID | Name      | Weight | Total Weight |
|------|----|-----------|--------|--------------|
| 1    | 5  | Alice     | 250    | 250          |
| 2    | 3  | Alex      | 350    | 600          |
| 3    | 6  | John Cena | 400    | 1000         | (last person to board)
| 4    | 2  | Marie     | 200    | 1200         | (cannot board)
| 5    | 4  | Bob       | 175    | ___          |
| 6    | 1  | Winston   | 500    | ___          |


In [ ]:
###your solution here

In [ ]:
def _case(rows, expected_name):
    q = pd.DataFrame(rows, columns=["personid", "personname", "weight", "turn"])
    got = sol_12(q)
    exp = pd.DataFrame({"personname": [expected_name]})
    assert_frame_equal(got.reset_index(drop=True), exp.reset_index(drop=True), check_dtype=False)


def run():
    cases = [
        ("example", lambda: _case(
            [(5, "Alice", 250, 1), (4, "Bob", 175, 5), (3, "Alex", 350, 2), (6, "John Cena", 400, 3), (1, "Winston", 500, 6), (2, "Marie", 200, 4)],
            "John Cena",
        )),
        ("exact_limit_last", lambda: _case(
            [(1, "A", 400, 1), (2, "B", 300, 2), (3, "C", 300, 3)],
            "C",
        )),
        ("second_exceeds", lambda: _case(
            [(1, "A", 900, 1), (2, "B", 200, 2), (3, "C", 50, 3)],
            "A",
        )),
        ("many_small_fit_until_one_exceeds", lambda: _case(
            [(1, "p1", 100, 1), (2, "p2", 200, 2), (3, "p3", 300, 3), (4, "p4", 250, 4), (5, "p5", 200, 5)],
            "p4",  # 100+200+300+250=850 ok; next 200 => 1050
        )),
        ("unordered_input_turn_matters", lambda: _case(
            [(2, "B", 600, 2), (1, "A", 500, 1), (3, "C", 100, 3)],
            "B",  # A (500) ok; B makes 1100 exceed -> last is A? wait 500+600=1100, so last is A
        )),
        ("turn_matters_fix", lambda: _case(
            [(2, "B", 600, 2), (1, "A", 500, 1), (3, "C", 100, 3)],
            "A",
        )),
        ("first_alone_at_limit", lambda: _case(
            [(1, "A", 1000, 1), (2, "B", 1, 2)],
            "A",
        )),
        ("weights_zero", lambda: _case(
            [(1, "A", 0, 1), (2, "B", 0, 2), (3, "C", 1000, 3)],
            "C",
        )),
        ("large_n", lambda: _case(
            [(i, f"p{i}", 100, i) for i in range(1, 20)],
            "p10",  # 10*100=1000, p11 would exceed
        )),
        ("nontrivial_mixed", lambda: _case(
            [(1, "A", 200, 1), (2, "B", 800, 2), (3, "C", 1, 3)],
            "B",
        )),
    ]

    failed = 0
    for name, f in cases:
        try:
            f()
        except Exception:
            failed += 1
    print(failed)


run()

### Exercise 13

Table: `Customer`


| Column Name   | Type    |
|---------------|---------|
| customer_id   | int     |
| name          | varchar |
| visited_on    | date    |
| amount        | int     |

In SQL,(customer_id, visited_on) is the primary key for this table.\
This table contains data about customer transactions in a restaurant.\
visited_on is the date on which the customer with ID (customer_id) has visited the restaurant.\
amount is the total paid by a customer.
 

You are the restaurant owner and you want to analyze a possible expansion (there will be at least one customer every day).

Compute the moving average of how much the customer paid in a seven days window (i.e., current day + 6 days before). average_amount should be rounded to two decimal places.

Return the result table ordered by visited_on in ascending order.

The result format is in the following example.

 

Example 1:

Input: 
Customer table:

| customer_id | name         | visited_on   | amount      |
|-------------|--------------|--------------|-------------|
| 1           | Jhon         | 2019-01-01   | 100         |
| 2           | Daniel       | 2019-01-02   | 110         |
| 3           | Jade         | 2019-01-03   | 120         |
| 4           | Khaled       | 2019-01-04   | 130         |
| 5           | Winston      | 2019-01-05   | 110         | 
| 6           | Elvis        | 2019-01-06   | 140         | 
| 7           | Anna         | 2019-01-07   | 150         |
| 8           | Maria        | 2019-01-08   | 80          |
| 9           | Jaze         | 2019-01-09   | 110         | 
| 1           | Jhon         | 2019-01-10   | 130         | 
| 3           | Jade         | 2019-01-10   | 150         | 

Output: 

| visited_on   | amount       | average_amount |
|--------------|--------------|----------------|
| 2019-01-07   | 860          | 122.86         |
| 2019-01-08   | 840          | 120            |
| 2019-01-09   | 840          | 120            |
| 2019-01-10   | 1000         | 142.86         |

Explanation: \
1st moving average from 2019-01-01 to 2019-01-07 has an average_amount of (100 + 110 + 120 + 130 + 110 + 140 + 150)/7 = 122.86\
2nd moving average from 2019-01-02 to 2019-01-08 has an average_amount of (110 + 120 + 130 + 110 + 140 + 150 + 80)/7 = 120\
3rd moving average from 2019-01-03 to 2019-01-09 has an average_amount of (120 + 130 + 110 + 140 + 150 + 80 + 110)/7 = 120\
4th moving average from 2019-01-04 to 2019-01-10 has an average_amount of (130 + 110 + 140 + 150 + 80 + 110 + 130 + 150)/7 = 142.86

In [ ]:
###your solution here

In [ ]:
def _norm(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "visitedon" in df.columns:
        df["visitedon"] = pd.to_datetime(df["visitedon"])
        df = df.sort_values(["visitedon"]).reset_index(drop=True)
    return df.reset_index(drop=True)


def _rename_avg(df: pd.DataFrame) -> pd.DataFrame:
    if "averageamount" not in df.columns and "average_amount" in df.columns:
        return df.rename(columns={"average_amount": "averageamount"})
    return df


def _case(rows, expected_rows, atol=1e-2):
    c = pd.DataFrame(rows, columns=["customerid", "name", "visitedon", "amount"])
    c["visitedon"] = pd.to_datetime(c["visitedon"])
    got = _rename_avg(sol_13(c))
    exp = pd.DataFrame(expected_rows, columns=["visitedon", "amount", "averageamount"])
    exp["visitedon"] = pd.to_datetime(exp["visitedon"])
    assert_frame_equal(_norm(got), _norm(exp), check_dtype=False, atol=atol, rtol=0)


def run():
    cases = [
        ("example", lambda: _case(
            [
                (1, "Jhon", "2019-01-01", 100),
                (2, "Daniel", "2019-01-02", 110),
                (3, "Jade", "2019-01-03", 120),
                (4, "Khaled", "2019-01-04", 130),
                (5, "Winston", "2019-01-05", 110),
                (6, "Elvis", "2019-01-06", 140),
                (7, "Anna", "2019-01-07", 150),
                (8, "Maria", "2019-01-08", 80),
                (9, "Jaze", "2019-01-09", 110),
                (1, "Jhon", "2019-01-10", 130),
                (3, "Jade", "2019-01-10", 150),
            ],
            [
                ("2019-01-07", 860, 122.86),
                ("2019-01-08", 840, 120.00),
                ("2019-01-09", 840, 120.00),
                ("2019-01-10", 1000, 142.86),
            ],
        )),
        ("exact_7_days_constant", lambda: _case(
            [(i, f"u{i}", f"2020-01-0{i}", 10) for i in range(1, 8)],
            [("2020-01-07", 70, 10.00)],
        )),
        ("8_days_constant", lambda: _case(
            [(i, f"u{i}", f"2020-01-0{i}", 10) for i in range(1, 9)],
            [("2020-01-07", 70, 10.00), ("2020-01-08", 70, 10.00)],
        )),
        ("multiple_customers_same_day", lambda: _case(
            [
                (1, "a", "2020-01-01", 10), (2, "b", "2020-01-01", 5),
                (1, "a", "2020-01-02", 10),
                (1, "a", "2020-01-03", 10),
                (1, "a", "2020-01-04", 10),
                (1, "a", "2020-01-05", 10),
                (1, "a", "2020-01-06", 10),
                (1, "a", "2020-01-07", 10),
            ],
            [("2020-01-07", 65, 9.29)],  # daily totals: 15,10,10,10,10,10,10 sum=65 avg=9.2857
        )),
        ("increasing_daily", lambda: _case(
            [(i, "x", f"2020-02-{i:02d}", i) for i in range(1, 11)],
            [
                ("2020-02-07", 28, 4.00),
                ("2020-02-08", 35, 5.00),
                ("2020-02-09", 42, 6.00),
                ("2020-02-10", 49, 7.00),
            ],
        )),
        ("negative_amounts", lambda: _case(
            [(i, "x", f"2020-03-{i:02d}", -10) for i in range(1, 8)],
            [("2020-03-07", -70, -10.00)],
        )),
        ("zero_amounts", lambda: _case(
            [(i, "x", f"2020-04-{i:02d}", 0) for i in range(1, 9)],
            [("2020-04-07", 0, 0.00), ("2020-04-08", 0, 0.00)],
        )),
        ("large_values", lambda: _case(
            [(i, "x", f"2020-05-{i:02d}", 1000000) for i in range(1, 8)],
            [("2020-05-07", 7000000, 1000000.00)],
        )),
        ("decimal_amounts", lambda: _case(
            [(i, "x", f"2020-06-{i:02d}", 0.1) for i in range(1, 8)],
            [("2020-06-07", 0.7, 0.10)],
            atol=1e-6,
        )),
        ("two_entries_last_day", lambda: _case(
            [(i, "x", f"2020-07-{i:02d}", 10) for i in range(1, 7)]
            + [(7, "x", "2020-07-07", 10), (8, "y", "2020-07-07", 5)],
            [("2020-07-07", 65, 9.29)],
        )),
    ]

    failed = 0
    for _, f in cases:
        try:
            f()
        except Exception:
            failed += 1
    print(failed)


run()

### Exercise 14

Table: `Movies`

| Column Name   | Type    |
|---------------|---------|
| movie_id      | int     |
| title         | varchar |

movie_id is the primary key (column with unique values) for this table.\
title is the name of the movie.\
Each movie has a unique title.\
Table: Users


| Column Name   | Type    |
|---------------|---------|
| user_id       | int     |
| name          | varchar |

user_id is the primary key (column with unique values) for this table\
The column 'name' has unique values.\
Table: MovieRating


| Column Name   | Type    |
|---------------|---------|
| movie_id      | int     |
| user_id       | int     |
| rating        | int     |
| created_at    | date    |

(movie_id, user_id) is the primary key (column with unique values) for this table.\
This table contains the rating of a movie by a user in their review.\
created_at is the user's review date. 
 

Write a solution to:

Find the name of the user who has rated the greatest number of movies. In case of a tie, return the lexicographically smaller user name.\
Find the movie name with the highest average rating in February 2020. In case of a tie, return the lexicographically smaller movie name.\
The result format is in the following example.

 

Example 1:

Input: \
Movies table:

| movie_id    |  title       |
|-------------|--------------|
| 1           | Avengers     |
| 2           | Frozen 2     |
| 3           | Joker        |

Users table:

| user_id     |  name        |
|-------------|--------------|
| 1           | Daniel       |
| 2           | Monica       |
| 3           | Maria        |
| 4           | James        |

MovieRating table:

| movie_id    | user_id      | rating       | created_at  |
|-------------|--------------|--------------|-------------|
| 1           | 1            | 3            | 2020-01-12  |
| 1           | 2            | 4            | 2020-02-11  |
| 1           | 3            | 2            | 2020-02-12  |
| 1           | 4            | 1            | 2020-01-01  |
| 2           | 1            | 5            | 2020-02-17  | 
| 2           | 2            | 2            | 2020-02-01  | 
| 2           | 3            | 2            | 2020-03-01  |
| 3           | 1            | 3            | 2020-02-22  | 
| 3           | 2            | 4            | 2020-02-25  | 

Output: 

| results      |
|--------------|
| Daniel       |
| Frozen 2     |

Explanation:
* Daniel and Monica have rated 3 movies ("Avengers", "Frozen 2" and "Joker") but Daniel is smaller lexicographically.\
* Frozen 2 and Joker have a rating average of 3.5 in February but Frozen 2 is smaller lexicographically.

In [ ]:
###your solution here

In [ ]:
def _norm(df: pd.DataFrame) -> pd.DataFrame:
    return df.reset_index(drop=True)


def _case(movies_rows, users_rows, ratings_rows, expected_results):
    movies = pd.DataFrame(movies_rows, columns=["movieid", "title"])
    users = pd.DataFrame(users_rows, columns=["userid", "name"])
    rating = pd.DataFrame(ratings_rows, columns=["movieid", "userid", "rating", "createdat"])
    rating["createdat"] = pd.to_datetime(rating["createdat"])
    got = sol_14(movies, users, rating)
    exp = pd.DataFrame({"results": expected_results})
    assert_frame_equal(_norm(got), _norm(exp), check_dtype=False)


def run():
    cases = [
        ("example", lambda: _case(
            [(1, "Avengers"), (2, "Frozen 2"), (3, "Joker")],
            [(1, "Daniel"), (2, "Monica"), (3, "Maria"), (4, "James")],
            [
                (1, 1, 3, "2020-01-12"),
                (1, 2, 4, "2020-02-11"),
                (1, 3, 2, "2020-02-12"),
                (1, 4, 1, "2020-01-01"),
                (2, 1, 5, "2020-02-17"),
                (2, 2, 2, "2020-02-01"),
                (2, 3, 2, "2020-03-01"),
                (3, 1, 3, "2020-02-22"),
                (3, 2, 4, "2020-02-25"),
            ],
            ["Daniel", "Frozen 2"],
        )),
        ("tie_user_count_lex_smallest", lambda: _case(
            [(1, "A"), (2, "B")],
            [(1, "Amy"), (2, "Bob")],
            [(1, 1, 5, "2020-02-10"), (2, 2, 5, "2020-02-10")],
            ["Amy", "A"],
        )),
        ("user_unique_most_ratings", lambda: _case(
            [(1, "A"), (2, "B"), (3, "C")],
            [(1, "U1"), (2, "U2")],
            [(1, 1, 3, "2020-02-01"), (2, 1, 4, "2020-02-02"), (3, 1, 5, "2020-02-03"), (1, 2, 5, "2020-02-04")],
            ["U1", "C"],  # Feb avg: A=(3+5)/2=4, B=4, C=5 => C
        )),
        ("movie_highest_avg_in_feb_only", lambda: _case(
            [(1, "A"), (2, "B")],
            [(1, "U")],
            [(1, 1, 5, "2020-01-10"), (2, 1, 4, "2020-02-10")],
            ["U", "B"],
        )),
        ("tie_movie_avg_lex_smallest", lambda: _case(
            [(1, "A"), (2, "B")],
            [(1, "U1"), (2, "U2")],
            [(1, 1, 4, "2020-02-01"), (1, 2, 2, "2020-02-02"), (2, 1, 3, "2020-02-03"), (2, 2, 3, "2020-02-04")],
            ["U1", "A"],  # both avgs=3.0 in Feb => A
        )),
        ("feb_2020_filter_strict", lambda: _case(
            [(1, "A"), (2, "B")],
            [(1, "U1")],
            [(1, 1, 5, "2020-02-29"), (2, 1, 1, "2020-03-01")],
            ["U1", "A"],
        )),
        ("multiple_ratings_same_movie_feb_avg_correct", lambda: _case(
            [(1, "A"), (2, "B")],
            [(1, "U1"), (2, "U2"), (3, "U3")],
            [(1, 1, 5, "2020-02-01"), (1, 2, 1, "2020-02-02"), (1, 3, 1, "2020-02-03"), (2, 1, 4, "2020-02-04")],
            ["U1", "B"],  # A avg=7/3=2.33, B avg=4
        )),
        ("ratings_outside_feb_do_not_affect_movie_pick", lambda: _case(
            [(1, "A"), (2, "B")],
            [(1, "U1")],
            [(1, 1, 1, "2020-02-10"), (2, 1, 5, "2020-01-10"), (2, 1, 5, "2020-03-10")],
            ["U1", "A"],
        )),
        ("user_count_all_time_not_feb_only", lambda: _case(
            [(1, "A")],
            [(1, "U1"), (2, "U2")],
            [(1, 1, 5, "2019-01-01"), (1, 1, 4, "2019-02-01"), (1, 2, 3, "2020-02-01")],
            ["U1", "A"],
        )),
        ("movie_title_lex_order_matters", lambda: _case(
            [(1, "AA"), (2, "A")],
            [(1, "U")],
            [(1, 1, 5, "2020-02-01"), (2, 1, 5, "2020-02-01")],
            ["U", "A"],  # tie avg => lex smaller "A"
        )),
    ]

    failed = 0
    for _, f in cases:
        try:
            f()
        except Exception:
            failed += 1
    print(failed)

run()